# Merge QLoRA Adapter into Base Model

This notebook merges the QLoRA adapter at `outputs/sft_qlora/final_adapter` into the base model `Qwen/Qwen2.5-14B-Instruct` and saves a standalone merged model to `outputs/merged_model` on Google Drive.

- **Expected GPU VRAM:** around 40 GB+ is recommended for safe merging.
- **Output:** merged model weights + tokenizer in `outputs/merged_model`.
- **Persistence:** output paths are symlinked to Google Drive.


## Setup

Mount Drive, infer workspace, clone/pull repo, and install minimum dependencies.

In [ ]:
from pathlib import Path
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_FALLBACK = '/content/drive/MyDrive/FinReasoningAI'

def _infer_notebook_workspace():
    """Parent folder of FinReasoningAI_Colab.ipynb on Drive (depth-limited, quick)."""
    root = Path('/content/drive/MyDrive')
    if not root.is_dir():
        return None
    candidates = []
    if (root / 'FinReasoningAI_Colab.ipynb').is_file():
        candidates.append(root.resolve())
    for child in sorted(root.iterdir()):
        if not child.is_dir():
            continue
        if (child / 'FinReasoningAI_Colab.ipynb').is_file():
            candidates.append(child.resolve())
        nested = child / 'FinReasoningAI'
        if nested.is_dir() and (nested / 'FinReasoningAI_Colab.ipynb').is_file():
            candidates.append(nested.resolve())
    uniq = []
    seen = set()
    for c in candidates:
        s = str(c)
        if s not in seen:
            seen.add(s)
            uniq.append(c)
    if len(uniq) == 1:
        return str(uniq[0])
    if len(uniq) > 1:
        print('[WARN] Multiple FinReasoningAI_Colab.ipynb paths on Drive; using DRIVE_FALLBACK.')
    return None

DRIVE_BASE = _infer_notebook_workspace() or DRIVE_FALLBACK
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive workspace: {DRIVE_BASE}')


In [ ]:
REPO_URL = 'https://github.com/juankim834/FinReasoningAI.git'

import os
import sys

WORKSPACE = DRIVE_BASE
if os.path.isdir(os.path.join(WORKSPACE, '.git')):
    PROJECT_DIR = WORKSPACE
else:
    PROJECT_DIR = os.path.join(WORKSPACE, 'FinReasoningAI')

if not os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
    os.makedirs(WORKSPACE, exist_ok=True)
    print(f'Cloning {REPO_URL} -> {PROJECT_DIR}')
    get_ipython().system(f'git clone {REPO_URL} {PROJECT_DIR}')
else:
    print(f'Repo already at {PROJECT_DIR}. Pulling latest...')
    get_ipython().system(f'cd {PROJECT_DIR} && git pull')

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')


In [ ]:
"""
Install only required dependencies (no torch reinstall).
Hard requirement: bitsandbytes >= 0.44.0.
"""

import importlib.metadata
import subprocess
import sys

REQUIRED = {
    'transformers': '4.41.0',
    'peft': '0.10.0',
    'bitsandbytes': '0.44.0',
    'accelerate': '0.30.0',
}

def _parse(v):
    out = []
    for p in v.split('.'):
        if p.isdigit():
            out.append(int(p))
        else:
            break
    while len(out) < 3:
        out.append(0)
    return tuple(out[:3])

def _installed(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None

to_install = []
for pkg, min_ver in REQUIRED.items():
    cur = _installed(pkg)
    if cur is None or _parse(cur) < _parse(min_ver):
        to_install.append(f'{pkg}>={min_ver}')
        status = 'MISSING' if cur is None else f'upgrade {cur} -> >= {min_ver}'
    else:
        status = f'ok ({cur})'
    print(f'{pkg:<14} {status}')

if to_install:
    print(f'\nInstalling {len(to_install)} package(s): {to_install}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + to_install)
    print('Install complete.')
else:
    print('\nAll required packages already satisfy minimum versions.')

bnb_ver = importlib.metadata.version('bitsandbytes')
if _parse(bnb_ver) < _parse('0.44.0'):
    raise RuntimeError(
        f'bitsandbytes {bnb_ver} is installed but >= 0.44.0 is required.\n'
        "Fix: pip install -U 'bitsandbytes>=0.44.0' then Runtime > Restart session."
    )

import torch
print(f'\nPyTorch version (unchanged): {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / (1024**3):.1f} GB')


In [ ]:
# Ensure project output directories are symlinked to Drive for persistence.
from pathlib import Path
import shutil
import os

def ensure_drive_symlink(rel_path: str):
    drive_path = Path(DRIVE_BASE) / rel_path
    local_path = Path(PROJECT_DIR) / rel_path
    drive_path.mkdir(parents=True, exist_ok=True)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.is_symlink():
        print(f'[OK] Symlink exists: {local_path} -> {local_path.resolve()}')
        return

    if local_path.exists():
        # Migrate existing local files into Drive folder, then replace with symlink.
        if local_path.is_dir():
            shutil.copytree(local_path, drive_path, dirs_exist_ok=True)
            shutil.rmtree(local_path)
        else:
            drive_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local_path, drive_path)
            local_path.unlink()

    os.symlink(drive_path, local_path)
    print(f'[OK] Linked: {local_path} -> {drive_path}')

for _rel in ['outputs/sft_qlora', 'outputs/merged_model']:
    ensure_drive_symlink(_rel)


In [ ]:
# Hugging Face login with Colab secret fallback.
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in using Colab secret HF_TOKEN.')
except Exception:
    print('HF_TOKEN secret not found. Falling back to interactive login...')
    login()


## Merge Adapter

Load base model in 4-bit NF4, attach LoRA adapter, dequantize to BF16, merge, and save.

In [ ]:
import importlib.util
from pathlib import Path
import torch
from peft import PeftModel
from src.model.load_model import load_model_and_tokenizer, DEFAULT_BNB_CONFIG, DEFAULT_MODEL_ID

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for merging this model.')

props = torch.cuda.get_device_properties(0)
total_vram_gb = props.total_memory / (1024**3)
print(f'GPU: {props.name} | Total VRAM: {total_vram_gb:.1f} GB')
if total_vram_gb < 50:
    print('[WARN] VRAM is below 50 GB. Merge can fail due to memory pressure (~40 GB+ needed).')

attn_impl = 'flash_attention_2' if importlib.util.find_spec('flash_attn') is not None else 'eager'
print(f'Attention backend: {attn_impl}')

print('Loading base model in 4-bit NF4...')
model, tokenizer = load_model_and_tokenizer(
    model_id=DEFAULT_MODEL_ID,
    bnb_config=DEFAULT_BNB_CONFIG,
    attn_implementation=attn_impl,
)

adapter_dir = Path('outputs/sft_qlora/final_adapter')
if not adapter_dir.exists():
    raise FileNotFoundError(f'Adapter path not found: {adapter_dir.resolve()}')

print('Dequantizing 4-bit model before merge...')
if hasattr(model, 'dequantize'):
    model = model.dequantize()
else:
    raise RuntimeError(
        'This transformers/bitsandbytes build does not expose model.dequantize(). '
        'Please upgrade and restart runtime.'
    )

# After dequantization, cast to BF16 for lower memory pressure during merge.
if getattr(model, 'dtype', None) != torch.bfloat16:
    model = model.to(torch.bfloat16)

print(f'Loading LoRA adapter from: {adapter_dir.resolve()}')
peft_model = PeftModel.from_pretrained(model, str(adapter_dir))

print('Merging adapter into base model...')
merged = peft_model.merge_and_unload()

merged_dir = Path('outputs/merged_model')
merged_dir.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(merged_dir))
tokenizer.save_pretrained(str(merged_dir))

print('\nSaved merged model artifacts:')
total_size = 0
for p in sorted(merged_dir.glob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / (1024**2)
        total_size += p.stat().st_size
        print(f'  {p.name:<40} {size_mb:9.2f} MB')
print(f'Total size: {total_size / (1024**3):.2f} GB')

print(f'\nLocal merged path: {merged_dir.resolve()}')
print(f'Drive merged path: {(Path(DRIVE_BASE) / "outputs/merged_model").resolve()}')
print('[OK] Merge complete.')
